⚠️ نکته:  ممکن است برای ویدیوهای چندزبانه کار نکند

In [ ]:
#@title ⚙️ Install
!pip install -q git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git
print("✅ Done!")

In [ ]:
#@title 📂 Upload
from google.colab import files
import subprocess, os

MEDIA_EXTS = ('.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg', '.opus',
              '.mp4', '.mkv', '.mov', '.webm', '.avi', '.m4v')
READY = False
SRT_FILE = MEDIA_FILE = None

def _pick(up):
    global SRT_FILE, MEDIA_FILE
    SRT_FILE = SRT_FILE or next((f for f in up if f.lower().endswith('.srt')), None)
    MEDIA_FILE = MEDIA_FILE or next((f for f in up if f.lower().endswith(MEDIA_EXTS)), None)

print("📄🎵 فایل زیرنویس (.srt) و فایل صوتی یا ویدیو را انتخاب کنید (می‌توانید هر دو را با هم انتخاب کنید):")
_pick(files.upload())

for _ in range(2):
    if SRT_FILE and MEDIA_FILE: break
    print("📄 هنوز فایل زیرنویس (.srt) را ندادید:" if not SRT_FILE else "🎵 هنوز فایل صوتی/ویدیو را ندادید:")
    _pick(files.upload())

if not SRT_FILE or not MEDIA_FILE:
    print("❌ هر دو فایل لازم است. این سلول را دوباره اجرا کنید.")
else:
    print(f"🔄 در حال آماده‌سازی صدا از {MEDIA_FILE} ...")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", MEDIA_FILE, "-vn", "-ac", "1",
                    "-ar", "16000", "_align_audio.wav"], check=True)
    mb = os.path.getsize("_align_audio.wav") / 1e6
    READY = True
    print(f"✅ آماده شد: زیرنویس = {SRT_FILE} | صدا = {MEDIA_FILE} ({mb:.0f} MB پس از تبدیل)")
    print("▶️ حالا سلول تنظیمات را اجرا کنید.")

In [ ]:
#@title 🎯 Settings & Sync (تنظیمات و هماهنگ‌سازی)
Words_Per_Line = 0        #@param {type:"integer"}
Balance_Lines = True      #@param {type:"boolean"}
Language = "eng"          #@param ["fas","eng","ara","tur","rus","spa","fra","deu","hin","urd","zho","jpn","kor"] {allow-input: true}
Hold_Seconds = 0.15       #@param {type:"number"}
Min_Duration = 0.4        #@param {type:"number"}
# Words_Per_Line = 0  یعنی تعداد کلمات هر دیالوگ همان‌طور که هست بماند (فقط زمان‌ها دقیق شوند)
# Words_Per_Line = 1  یعنی هر کلمه یک دیالوگ

import subprocess, sys, os, re, math
from google.colab import files
from IPython.display import display, HTML

def _fmt(sec):
    ms = int(round(max(sec, 0) * 1000))
    h, r = divmod(ms, 3600000); m, r = divmod(r, 60000); s, ms = divmod(r, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

def _read_srt(path):
    raw = open(path, "rb").read()
    for enc in ("utf-8-sig", "utf-16", "cp1256", "latin-1"):
        try: txt = raw.decode(enc); break
        except Exception: continue
    txt = txt.replace("\r\n", "\n").replace("\r", "\n").strip()
    out = []
    for b in re.split(r"\n\s*\n", txt):
        lines = b.split("\n")
        i = next((k for k, l in enumerate(lines) if "-->" in l), None)
        if i is None: continue
        m = re.match(r"\s*(\d+):(\d+):(\d+)[,.](\d+)\s*-->\s*(\d+):(\d+):(\d+)[,.](\d+)", lines[i])
        if not m: continue
        g = m.groups()
        tt = lambda h, mi, s, ms: int(h) * 3600 + int(mi) * 60 + int(s) + float("0." + ms)
        text = re.sub(r"<[^>]+>|\{[^}]+\}", "", " ".join(lines[i + 1:]))
        text = re.sub(r"\s+", " ", text).strip()
        if text: out.append({"s": tt(*g[:4]), "e": tt(*g[4:]), "t": text})
    return out

def _sizes(m, n, balance):
    if n <= 0 or m <= n: return [m]
    if balance:
        k = math.ceil(m / n); base, extra = divmod(m, k)
        return [base + 1 if i < extra else base for i in range(k)]
    sz = [n] * (m // n)
    if m % n: sz.append(m % n)
    return sz

def show_subscribe_banner():
    image_url = 'https://huggingface.co/Toolsai/dubtest/resolve/main/newgolden.png'
    youtube_channel_url = 'https://youtube.com/@aigolden'
    display(HTML(f"""
    <div style="text-align: center; border: 2px solid #e0e0e0; padding: 15px; border-radius: 12px; background-color: #f9f9f9; max-width: 350px; margin: auto;">
        <a href="{youtube_channel_url}" target="_blank" title="رفتن به کانال یوتیوب AIGOLDEN">
            <img src="{image_url}" alt="AIGOLDEN YouTube Channel" style="max-width: 100%; height: auto; border-radius: 8px;">
        </a>
        <p style="font-size: 16px; font-family: 'Vazir', sans-serif; margin-top: 15px; color: #333;">
            برای مشاهده آموزش‌های بیشتر، ما را در یوتیوب دنبال کنید.
        </p>
        <a href="{youtube_channel_url}" target="_blank" style="text-decoration: none; display: inline-block; background-color: #FF0000; color: white; padding: 10px 20px; border-radius: 8px; font-weight: bold; font-family: 'Vazir', sans-serif; margin-top: 10px;">
            🚀 دنبال کردن در یوتیوب
        </a>
    </div>"""))

if not globals().get("READY"):
    print("❌ ابتدا سلول Upload را اجرا کنید.")
else:
    try:
        import torch
        from ctc_forced_aligner import (load_audio, load_alignment_model, generate_emissions,
                                        preprocess_text, get_alignments, get_spans, postprocess_results)

        cues = _read_srt(SRT_FILE)
        words_all = [w for c in cues for w in c["t"].split()]
        assert words_all, "هیچ متنی در فایل SRT پیدا نشد."
        print(f"📄 {len(cues)} دیالوگ و {len(words_all)} کلمه خوانده شد.")

        cuda = torch.cuda.is_available()
        if not cuda: print("⚠️ GPU فعال نیست؛ روی CPU کند است.")
        device = "cuda" if cuda else "cpu"
        model, tok = load_alignment_model(device, dtype=torch.float16 if cuda else torch.float32)
        wav = load_audio("_align_audio.wav", model.dtype, model.device)
        print("⏳ در حال تحلیل صدا و پیدا کردن زمان دقیق کلمات...")
        emissions, stride = generate_emissions(model, wav, batch_size=16 if cuda else 4)

        res, err = None, None
        for lang in dict.fromkeys([Language, "fas", "pes"]):
            try:
                res = preprocess_text(" ".join(words_all), romanize=True, language=lang); break
            except Exception as e:
                err = e
        if res is None: raise RuntimeError(f"کد زبان پذیرفته نشد: {err}")
        tokens_starred, text_starred = res
        segments, scores, blank = get_alignments(emissions, tokens_starred, tok)
        spans = get_spans(tokens_starred, segments, blank)
        wts = postprocess_results(text_starred, spans, stride, scores)
        if len(wts) != len(words_all):
            raise RuntimeError(f"تعداد کلمات نخواند ({len(wts)} به‌جای {len(words_all)})")

        # ---- زمان هر کلمه (با بررسی اعتبار) ----
        k, per_cue, flags = 0, [], []
        for c in cues:
            ws = c["t"].split(); n = len(ws)
            w = [[a, float(p["start"]), float(p["end"])] for a, p in zip(ws, wts[k:k + n])]; k += n
            st_ = [x[1] for x in w]
            flags.append(all(b >= a for a, b in zip(st_, st_[1:])) and (n == 1 or st_[-1] - st_[0] >= 0.05 * (n - 1)))
            per_cue.append(w)
        # دیالوگ‌های نامعتبر: تخمین بین آخرین کلمه‌ی سالم قبل و اولین کلمه‌ی سالم بعد
        total_dur = float(subprocess.check_output(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                                                   "-of", "csv=p=0", "_align_audio.wav"]))
        i = 0
        while i < len(cues):
            if flags[i]: i += 1; continue
            j = i
            while j < len(cues) and not flags[j]: j += 1
            left = per_cue[i - 1][-1][2] if i > 0 else 0.0
            right = per_cue[j][0][1] if j < len(cues) else total_dur
            run = [x for q in range(i, j) for x in per_cue[q]]
            if right <= left: right = left + 0.3 * len(run)
            tot = sum(len(x[0]) + 1 for x in run); t0 = left
            for x in run:
                d = (right - left) * (len(x[0]) + 1) / tot; x[1], x[2] = t0, t0 + d; t0 += d
            i = j
        bad = flags.count(False)

        prev = 0.0                       # زمان‌ها باید صعودی باشند
        for w in per_cue:
            for x in w:
                x[1] = max(x[1], prev); x[2] = max(x[2], x[1] + 0.02); prev = x[1]

        # ---- گروه‌بندی با تعداد کلمه‌ی دلخواه ----
        out = []
        for w in per_cue:
            i = 0
            for sz in _sizes(len(w), int(Words_Per_Line), Balance_Lines):
                g = w[i:i + sz]; i += sz
                out.append({"s": g[0][1], "e": g[-1][2], "t": " ".join(x[0] for x in g)})
        for i, c in enumerate(out):
            nxt = out[i + 1]["s"] if i + 1 < len(out) else 1e9
            e = min(c["e"] + Hold_Seconds, nxt)
            if e - c["s"] < Min_Duration: e = min(c["s"] + Min_Duration, nxt)
            c["e"] = max(e, c["s"] + 0.05)

        OUT_FILE = SRT_FILE.rsplit(".", 1)[0] + "_precise.srt"
        with open(OUT_FILE, "w", encoding="utf-8") as f:
            for i, c in enumerate(out, 1):
                f.write(f"{i}\n{_fmt(c['s'])} --> {_fmt(c['e'])}\n{c['t']}\n\n")

        drift = [abs(per_cue[i][0][1] - cues[i]["s"]) for i in range(len(cues))]
        print(f"✅ {len(out)} دیالوگ با زمان‌بندی دقیق ساخته شد → {OUT_FILE}")
        print(f"   اختلاف با زمان‌های قبلی: میانگین {sum(drift)/len(drift):.2f}s ، بیشترین {max(drift):.2f}s")
        if bad: print(f"   ⚠️ برای {bad} دیالوگ نتیجه‌ی تراز نامعتبر بود و تخمین استفاده شد.")
        print("\n▶️ نمونه:")
        for c in out[:5]:
            print(f"   {_fmt(c['s'])} → {_fmt(c['e'])}   {c['t']}")
        print(f"\n📥 در حال دانلود: {OUT_FILE}")
        files.download(OUT_FILE)
        show_subscribe_banner()
    except Exception as e:
        print("⚠️ هماهنگ‌سازی ناموفق بود.")
        print("   علت:", repr(e))